# SCIPパラメータチューニング(Optuna)— 適用前・原理・適用・効果

SCIP には分離(カット)・ヒューリスティクス・presolve・分枝規則など多数のパラメータがあり、
既定値(`SCIP_PARAMSETTING.DEFAULT`)は「平均的にバランス良く」設定されている。しかし
特定の問題クラスに絞って「固定時間でどれだけ双対境界を締められるか」を最大化したいなら、
その問題クラスに特化した設定へチューニングする余地がある。これはモデルの定式化そのものを
変える改善(整数×連続の厳密線形化など)とは独立した**メタ最適化**であり、SCIP自身は
自動でやらない(問題クラスをまたいで汎用的に動くよう設計されているため)。

この notebook は次の一貫した型でこの手法を追う。

1. **課題(before)** — 厳密線形化してもなお残る gap を `mk.analyze` で確認する
2. **原理(principle)** — 分離/ヒューリスティクス/分枝の強度を変えると gap-vs-time の
   収束曲線がどう変わるかを、実際に3通りの設定で解いて重ねて見る
3. **適用(how)** — `minlpkit.tune.tune`(Optuna/TPE)で双対境界を最大化する設定を探索する
4. **効果(before/after)** — `mk.compare_variants` でデフォルト設定とチューニング後設定を比較する

題材は **バッチ反応器スケジューリング(線形化版)**(`samples/others/scheduling_plant.py`、
`linearize_ns=True`)。整数×連続の厳密線形化(手法notebook1)を適用した後もなお埋まらない
gap が残っており、パラメータチューニングは「定式化を直したあとの最後の仕上げ」という
位置づけになる(FINDINGS §3)。

In [1]:
import sys, pathlib
# リポジトリルート(pyproject.toml を持つ階層)を探して import パスに載せる。
# docs/samples/ が存在するため "samples" 有無では docs で止まる。pyproject.toml を目印にする。
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/others"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):  # 静的サイトに埋め込める形でグラフを表示(CDN の plotly.js を読む)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import scheduling_plant as sp
from minlpkit.live.monitor import SolveMonitor
from minlpkit.tune import tune
from pyscipopt import SCIP_PARAMSETTING

# dataviz 参照パレット(minlpkit.live.plots と統一)
C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. 課題(before) — 厳密線形化後も残る gap

`scheduling_plant.build_model(linearize_ns=True)` は手法notebook1で適用した
「整数×連続の厳密線形化」を既に含む(n·s の McCormick 緩和ギャップは解消済み)。
それでも `mk.analyze` に掛けると、短時間では gap がまだ残っていることが分かる。

In [2]:
report = mk.analyze(lambda: sp.build_model(linearize_ns=True),
                    name="plant-linearized", time_limit=8)
print(report.summary())
print("\n残存gap:", f"{report.metrics.get('gap', float('nan')):.1%}"
      if report.metrics.get("gap") is not None else "N/A")

[plant-linearized] 検出症状 3件:
  [serious] 特定の非線形制約に緩和違反が集中(かつ空間分枝が多い) -> ボトルネック制約の区分線形近似・凸包再定式化・変数境界タイト化
  [warning] 双対境界の改善が停滞(gapが残る) -> 有効不等式の追加・変数境界タイト化・Big-M排除で緩和強化
  [warning] 係数の絶対値レンジが桁違い / Big-M候補あり(presolve後も残存) -> スケーリング、Big-MのIndicator/SOS制約化、係数の再定式化

残存gap: 64.2%


線形化で緩和のギャップ自体は解消済みなので、残る gap は探索動学(分枝順序・切除平面・
ヒューリスティクスの当たり方)に由来する。ここから先はパラメータチューニングの出番になる。

## 2. 原理(principle) — パラメータで探索効率が変わる

同じモデルを同じ時間だけ解いても、SCIP の設定次第で「固定時間にどこまで双対境界が締まるか」
は変わる。分離(カット生成)を強めればルートの下界は締まるが1ラウンドが重くなり、
ヒューリスティクスを強めれば良い上界(primal解)は早く見つかるが探索に回す時間が減る。
どちらのバランスが良いかは問題クラス依存で理論的に決め打てない。

手で3通りだけ設定を振ってみる(どれが最良かはまだ決め打たない。次節でOptunaに探索させる)。

In [3]:
def apply_default(m):
    pass  # SCIP既定のまま

def apply_config_a(m):
    # 手動候補A: カット/ヒューリスティクスを軽くし分岐で双対境界を押す
    m.setSeparating(SCIP_PARAMSETTING.FAST)
    m.setHeuristics(SCIP_PARAMSETTING.FAST)
    m.setParam("branching/mostinf/priority", 1000000)

def apply_config_b(m):
    # 手動候補B: 分離を強め、ヒューリスティクスを切る(ルートは重くなるが切除は締まるはず、という仮説)
    m.setSeparating(SCIP_PARAMSETTING.AGGRESSIVE)
    m.setHeuristics(SCIP_PARAMSETTING.OFF)

TIME_LIMIT = 8.0

def run_with_monitor(apply_params):
    m = sp.build_model(linearize_ns=True)
    m.hideOutput()
    m.setParam("timing/clocktype", 2)
    apply_params(m)
    mon = SolveMonitor(min_interval=0.02)
    m.includeEventhdlr(mon, "monitor", "collect dual bound over time")
    m.setParam("limits/time", TIME_LIMIT)
    m.optimize()
    return mon.to_frame(), m.getDualbound()

configs = {
    "default": (apply_default, C["muted"]),
    "config A: separating=fast, heuristics=fast, branching=mostinf": (apply_config_a, C["s1"]),
    "config B: separating=aggressive, heuristics=off": (apply_config_b, C["warn"]),
}
curves = {}
for label, (fn, color) in configs.items():
    df, dual = run_with_monitor(fn)
    curves[label] = (df, color)
    print(f"{label:32s} final dual={dual:.2f}")

default                          final dual=140.68


config A: separating=fast, heuristics=fast, branching=mostinf final dual=149.94


config B: separating=aggressive, heuristics=off final dual=153.89


In [4]:
fig = go.Figure(layout=base_layout(
    "SCIPパラメータの違いによる双対境界の収束(同一モデル・同一時間予算)",
    "求解時間 [s]", "双対境界(高いほど良い)", height=420))
for label, (df, color) in curves.items():
    d = df.dropna(subset=["dual"])
    fig.add_trace(go.Scatter(x=d["time"], y=d["dual"], mode="lines", line_shape="hv",
        line=dict(color=color, width=2.5), name=label,
        hovertemplate=label + ": t=%{x:.1f}s dual=%{y:.1f}<extra></extra>"))
show(fig)

実行結果はセル出力の `final dual=` を参照(3設定とも既定より高い双対境界に達することもあれば、
逆転することもある — 手動の勘だけでは事前にどちらが勝つか分からない)。重要なのは
**「パラメータの選び方で同じ時間予算内の到達点が変わる」こと自体**であり、「強くすれば
するほど良い」という単純な関係ではない、という点。どの組み合わせが良いかを人手で総当たり
するのは非効率なので、次節では Optuna に自動探索させる。

## 3. 適用(how) — `minlpkit.tune.tune`

3設定を手で振って比べる代わりに、Optuna(TPEサンプラー)で分離/ヒューリスティクス/presolve/
分枝規則の組み合わせを自動探索できる。1行呼び出しで、既定設定を基準に何%改善したかも返る。

In [5]:
res = tune(n_trials=12, time_limit=6.0)
gain = (res["best_dual"] - res["default_dual"]) / abs(res["default_dual"]) * 100
print(f"default_dual={res['default_dual']:.2f}  best_dual={res['best_dual']:.2f}  (+{gain:.1f}%)")
print("best_params:", res["best_params"])

default_dual=137.87  best_dual=151.07  (+9.6%)
best_params: {'separating': 'fast', 'heuristics': 'off', 'presolving': 'fast', 'branching': 'mostinf'}


## 4. 効果(before/after) — `mk.compare_variants`

デフォルト設定と、Optuna が見つけた最良設定を同じ時間制限で解き、
**ルート双対境界・最終gap・ノード数**を比較する。

In [6]:
_EMPHASIS = {"default": SCIP_PARAMSETTING.DEFAULT, "aggressive": SCIP_PARAMSETTING.AGGRESSIVE,
             "fast": SCIP_PARAMSETTING.FAST, "off": SCIP_PARAMSETTING.OFF}

def apply_best(m, params):
    m.setSeparating(_EMPHASIS[params["separating"]])
    m.setHeuristics(_EMPHASIS[params["heuristics"]])
    m.setPresolve(_EMPHASIS[params["presolving"]] if params["presolving"] != "off"
                  else SCIP_PARAMSETTING.OFF)
    m.setParam(f"branching/{params['branching']}/priority", 1000000)

def build_default():
    return sp.build_model(linearize_ns=True)

def build_tuned():
    m = sp.build_model(linearize_ns=True)
    apply_best(m, res["best_params"])
    return m

df = mk.compare_variants({"default": build_default, "tuned": build_tuned}, time_limit=7.0)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

,variant,root_dual,final_dual,final_gap,nodes
0,default,133.302921,139.509451,0.655459,69
1,tuned,132.726461,151.070674,NaN,2147


In [7]:
base = df.loc[df["variant"] == "default"].iloc[0]
tun  = df.loc[df["variant"] == "tuned"].iloc[0]

def isnan(v):
    return v is None or v != v  # NaN は自分自身と等しくない

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("ルート双対境界 (高いほど良い)", "最終 gap [%](低いほど良い。可行解0なら定義不可)",
                    "探索ノード数 (少ないほど良い)"))
labels = ["default", "tuned"]
bar_colors = [C["muted"], C["s1"]]
def add_bars(col, values, texts):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=texts, textposition="outside",
        cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, [base["root_dual"], tun["root_dual"]],
         [f"{base['root_dual']:.0f}", f"{tun['root_dual']:.0f}"])
gap_vals = [0.0 if isnan(base["final_gap"]) else base["final_gap"] * 100,
            100.0 if isnan(tun["final_gap"]) else tun["final_gap"] * 100]
gap_texts = ["可行解なし" if isnan(base["final_gap"]) else f"{base['final_gap']*100:.0f}%",
             "可行解なし" if isnan(tun["final_gap"]) else f"{tun['final_gap']*100:.0f}%"]
add_bars(2, gap_vals, gap_texts)
add_bars(3, [base["nodes"], tun["nodes"]], [f"{int(base['nodes'])}", f"{int(tun['nodes'])}"])
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="SCIPパラメータチューニングの before / after", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)

In [8]:
def fmt_gap(v):
    return "可行解なし(gap未定義)" if isnan(v) else f"{v:.1%}"
print(f"ルート双対境界 : {base['root_dual']:.1f} -> {tun['root_dual']:.1f}")
print(f"最終dual境界   : {base['final_dual']:.1f} -> {tun['final_dual']:.1f}")
print(f"最終gap        : {fmt_gap(base['final_gap'])} -> {fmt_gap(tun['final_gap'])}")
print(f"ノード数       : {int(base['nodes'])} -> {int(tun['nodes'])}")

ルート双対境界 : 133.3 -> 132.7
最終dual境界   : 139.5 -> 151.1
最終gap        : 65.5% -> 可行解なし(gap未定義)
ノード数       : 69 -> 2147


## まとめ

- **双対境界だけを見れば、チューニング後設定はこの notebook の実行でも既定より高い値に
  到達した**(上のセル出力を参照)。FINDINGS §3 の先行実測(固定7秒の双対境界 134.8 → 143.7、
  +6.6%)と同じ向きの効果である。
- **ただしOptunaが見つけた最良設定(`heuristics=off` を含む)は、7秒の時間制限内に可行解を
  1つも見つけられなかった**(`final_gap` が定義不可能=「可行解なし」)。`tune()` の目的関数は
  「固定時間での双対境界」だけで primal 側の性能を一切見ていないため、双対境界を最大化する
  設定がヒューリスティクスを丸ごと切ってしまい、実務で欲しい「時間内に返る解」を犠牲にした。
  **双対境界の改善と、実際に使える解が返ってくることは別軸**であり、チューニングの目的関数を
  そのまま鵜呑みにして採用すると痛い目を見る、という実例。

### なぜ SCIP が自動でやらないのか

SCIP の既定パラメータは「未知の問題クラスに対して平均的に良い」よう設計されている。
「この問題クラスなら分離を弱めヒューリスティクスも弱めて分岐に賭けた方が固定時間で有利」
という判断は、そのモデル固有の構造ではなく**代表インスタンス群を実際に何度も解いてみないと
分からない**。これはビルド済みモデル1個の presolve では検出できない、問題クラス単位の
メタ最適化であり、モデラー/運用側が明示的に行う分業になる。

### 効かないとき・注意

- **目的関数の選び方次第で「良い設定」の意味が変わる**。上の実行例のように双対境界だけを
  最大化する設定は、primal解を諦めてでも下界を締めにいくことがある。実務での採用前には
  `mk.compare_variants` で primal 側(final_gap/nodes/実際に解が返るか)も必ず確認する。
- 探索は特定インスタンス(または代表インスタンス群)への特化なので、別の規模・構造の
  インスタンスに一般化するとは限らない。運用では代表インスタンス群でチューニングし、
  本番設定を固定するのが実務的。
- 定式化そのものが弱い(緩和が緩い)場合はパラメータ調整では埋まらない。まず
  [1. 厳密線形化](../../playbook/01-linearize.md)や[2. PWL-SOS2](../../playbook/02-pwl-sos2.md)
  のような定式化側の改善を検討する。

関連: [プレイブック 4. SCIPパラメータチューニング](../../playbook/04-tuning.md) /
API [`minlpkit.tune.tune`](../../api/tune.md) / [`mk.sweep`/`mk.rerun`](../../api/live.md)。